# Session 20: Final Capstone — Employee Attrition Pipeline

**Applied Machine Learning Using Python**  
**Duration**: 4 Hours (TL20)  
**Level**: Integrative (All Sessions)

---

## Objective
Build a complete ML pipeline that integrates techniques from 8+ previous sessions:
1. EDA & Data Cleaning (Sessions 12-13)
2. Handling Class Imbalance (Session 7)
3. Ensemble Learning (Sessions 9-10)
4. Model Evaluation (Session 10)
5. Explainability (Sessions 17, 19)
6. Ethical Auditing (Session 18)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
print("Libraries loaded.")

---
## Phase 1: Load & Explore the Data (Sessions 12-13)

First, generate the dataset by running `code/01_generate_data.py`, then load it here.

In [ ]:
df = pd.read_csv('../data/employee_attrition.csv')
print(f"Shape: {df.shape}")
display(df.head())
display(df.describe())

In [ ]:
# YOUR TASK: Check for missing values
# Hint: df.isnull().sum()
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0])

In [ ]:
# YOUR TASK: Visualize the class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class balance
df['Attrition'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Class Distribution')
axes[0].set_xticklabels(['Stayed', 'Left'], rotation=0)

# Salary by Gender
df.boxplot(column='MonthlySalary', by='Gender', ax=axes[1])
axes[1].set_title('Salary by Gender (Check for Pay Gap)')

plt.tight_layout()
plt.show()

---
## Phase 2: Data Cleaning (Sessions 12-13)

Handle missing values and outliers before modeling.

In [ ]:
# YOUR TASK: Impute missing values with median
numeric_cols_with_missing = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols_with_missing:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"Filled '{col}' NaNs with median = {median_val:.1f}")

print(f"\nRemaining missing: {df.isnull().sum().sum()}")

In [ ]:
# YOUR TASK: Detect and cap outliers in OvertimeHours using IQR
Q1 = df['OvertimeHours'].quantile(0.25)
Q3 = df['OvertimeHours'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

outliers_before = (df['OvertimeHours'] > upper_bound).sum()
df['OvertimeHours'] = df['OvertimeHours'].clip(upper=upper_bound)
print(f"Capped {outliers_before} overtime outliers at {upper_bound:.0f} hours")

---
## Phase 3: Feature Engineering & Train/Test Split (Sessions 1-2, 7)

In [ ]:
# YOUR TASK: Encode Gender and Department
df['Gender_Code'] = df['Gender'].map({'Male': 1, 'Female': 0})
dept_dummies = pd.get_dummies(df['Department'], prefix='Dept', drop_first=True)
df = pd.concat([df, dept_dummies], axis=1)

# Define features
exclude = ['EmployeeID', 'Gender', 'Department', 'Attrition']
feature_cols = [c for c in df.columns if c not in exclude]

X = df[feature_cols].values
y = df['Attrition'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(f"Features: {len(feature_cols)}")
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Train attrition rate: {y_train.mean():.1%}")

In [ ]:
# YOUR TASK: Apply SMOTE to balance the training set
try:
    from imblearn.over_sampling import SMOTE
    smote = SMOTE(random_state=42)
    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
    print(f"SMOTE: {len(X_train)} -> {len(X_train_bal)} samples")
    print(f"New attrition rate: {y_train_bal.mean():.1%}")
except ImportError:
    print("imbalanced-learn not installed. Using original data with class_weight='balanced'.")
    X_train_bal, y_train_bal = X_train, y_train

---
## Phase 4: Model Training & Evaluation (Sessions 9-10)

In [ ]:
# YOUR TASK: Train Random Forest and Gradient Boosting, compare ROC-AUC
rf = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42)
rf.fit(X_train_bal, y_train_bal)
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

gb = GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
gb.fit(X_train_bal, y_train_bal)
gb_auc = roc_auc_score(y_test, gb.predict_proba(X_test)[:, 1])

print(f"Random Forest     ROC-AUC: {rf_auc:.3f}")
print(f"Gradient Boosting ROC-AUC: {gb_auc:.3f}")

best = rf if rf_auc >= gb_auc else gb
best_name = 'Random Forest' if rf_auc >= gb_auc else 'Gradient Boosting'
print(f"\nSelected: {best_name}")

In [ ]:
# Full evaluation of the best model
y_pred = best.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Stayed', 'Left']))

# Confusion Matrix visualization
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=['Stayed', 'Left'], yticklabels=['Stayed', 'Left'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'{best_name} — Confusion Matrix')
plt.tight_layout()
plt.show()

---
## Phase 5: Explainability (Sessions 17 & 19)

In [ ]:
# YOUR TASK: Plot the top 10 most important features
importances = best.feature_importances_
sorted_idx = np.argsort(importances)[-10:]

plt.figure(figsize=(8, 5))
plt.barh([feature_cols[i] for i in sorted_idx], importances[sorted_idx], color='#e74c3c')
plt.xlabel('Feature Importance')
plt.title(f'{best_name} — Top 10 Attrition Predictors')
plt.tight_layout()
plt.show()

print(f"\nTop predictor: {feature_cols[sorted_idx[-1]]}")
print("HR should focus retention efforts on employees with extreme values in this feature.")

---
## Phase 6: Ethical Audit (Session 18)

In [ ]:
# YOUR TASK: Check Disparate Impact by Gender
gender_idx = feature_cols.index('Gender_Code')
test_genders = X_test[:, gender_idx]

male_leave_rate = y_pred[test_genders == 1].mean()
female_leave_rate = y_pred[test_genders == 0].mean()

print(f"Male 'At Risk' rate:   {male_leave_rate:.1%}")
print(f"Female 'At Risk' rate: {female_leave_rate:.1%}")

if male_leave_rate >= female_leave_rate:
    ratio = female_leave_rate / male_leave_rate
else:
    ratio = male_leave_rate / female_leave_rate

print(f"\nDisparate Impact Ratio: {ratio:.3f}")
print(f"Result: {'PASSED (>= 0.80)' if ratio >= 0.80 else 'FAILED (< 0.80) — Investigate proxy bias!'}")

# Check if MonthlySalary is acting as a proxy for gender
salary_idx = feature_cols.index('MonthlySalary')
print(f"\nMonthlySalary importance: {importances[salary_idx]:.4f}")
print(f"Gender_Code importance:   {importances[gender_idx]:.4f}")

---
## Conclusion

You've built a complete, production-grade ML pipeline from raw data to ethical audit.

This single notebook integrates:
- **EDA** (Sessions 12-13)
- **Imbalanced Data** (Session 7)
- **Ensemble Learning** (Sessions 9-10)
- **Model Evaluation** (Session 10)
- **Explainability** (Sessions 17, 19)
- **Ethical AI** (Session 18)

**Congratulations on completing the Applied Machine Learning Using Python curriculum!**